<a href="https://colab.research.google.com/github/muneerh-fa/Sdaia-agentic-ai-systems-course/blob/main/assignments/assignment3/assignment3_supervisor_pattern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — Supervisor Pattern with Sub-Agents

## Use Case: Online Shopping Support Assistant

## Overview

This notebook implements a different use case for the supervisor pattern.

The system is an **Online Shopping Support Assistant** with:
- an **Order Agent**
- a **Return Agent**
- a **Supervisor Agent**

The supervisor handles user requests by assigning each subtask to the appropriate specialized sub-agent.

In [11]:
%pip install -q langchain langchain_openai

## Load OpenRouter API Key

This cell loads the OpenRouter API key from Google Colab Secrets.

In [12]:
import os

try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()
    print("OPENROUTER_API_KEY loaded successfully")
except Exception as e:
    print("Error loading API key:", e)

OPENROUTER_API_KEY loaded successfully


## Create the Language Model

This model will be used by the sub-agents and the supervisor agent.

In [13]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

## Define the Low-Level Tools

These tools represent the basic operations used by the specialized agents.

In [14]:
from langchain.tools import tool

@tool
def track_order(order_id: str) -> str:
    """Track an order by its ID and return the shipping status."""
    fake_orders = {
        "1001": "Order 1001 is shipped and will arrive tomorrow.",
        "1002": "Order 1002 is still being prepared in the warehouse.",
        "1003": "Order 1003 has been delivered successfully."
    }
    return fake_orders.get(order_id, f"No order found with ID {order_id}.")

@tool
def check_return_policy(product_name: str, days_since_delivery: int) -> str:
    """Check whether a product can be returned based on the return policy."""
    if days_since_delivery <= 14:
        return f"{product_name} is eligible for return because it is within the 14-day return window."
    return f"{product_name} is not eligible for return because it is outside the 14-day return window."

@tool
def create_return_request(order_id: str, product_name: str) -> str:
    """Create a return request for a product in an order."""
    return f"Return request created for {product_name} in order {order_id}."

## Create the Order Agent

This agent is specialized in tracking orders and shipment updates.

In [15]:
from langchain.agents import create_agent

ORDER_AGENT_PROMPT = """
You are an order support assistant.
Your role is to help users track orders and check shipping status.

Use the track_order tool whenever needed.
Always provide a clear and concise final answer.
"""

order_agent = create_agent(
    model=model,
    tools=[track_order],
    system_prompt=ORDER_AGENT_PROMPT
)

## Create the Return Agent

This agent is specialized in checking return eligibility and creating return requests.

In [16]:
RETURN_AGENT_PROMPT = """
You are a returns assistant.
Your role is to help users with return eligibility and return requests.

Use check_return_policy when the user asks whether an item can be returned.
Use create_return_request when the user wants to start a return.
Always provide a clear and concise final answer.
"""

return_agent = create_agent(
    model=model,
    tools=[check_return_policy, create_return_request],
    system_prompt=RETURN_AGENT_PROMPT
)

## Wrap the Sub-Agents as Tools

The supervisor will not directly use the low-level tools.
Instead, it will call the specialized sub-agents as higher-level tools.

In [17]:
from langchain.tools import tool
from langchain.messages import HumanMessage

@tool
def manage_order(order_id: str) -> str:
    """Handle order-related requests using the order agent."""
    result = order_agent.invoke(
        {"messages": [HumanMessage(content=f"Track order {order_id}")]}
    )
    return result["messages"][-1].content

@tool
def manage_return(product_name: str, days_since_delivery: int) -> str:
    """Handle return-related requests using the return agent."""
    result = return_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Can I return {product_name} after {days_since_delivery} days?"
                )
            ]
        }
    )
    return result["messages"][-1].content

## Create the Supervisor Agent

The supervisor receives the user's request, decides which sub-agent should handle each part, and combines the results.

In [18]:
SUPERVISOR_PROMPT = """
You are an online shopping support supervisor.

Your job is to coordinate specialized sub-agents to solve user requests.

Available tools:
- manage_order(order_id): for order tracking and shipping questions
- manage_return(product_name, days_since_delivery): for return eligibility questions

If the user asks about tracking an order, use manage_order.
If the user asks whether an item can be returned, use manage_return.
If the request includes multiple tasks, use multiple tools in sequence.

Then combine the results into one final helpful response.
Keep the answer short and clear.
"""

supervisor_agent = create_agent(
    model=model,
    tools=[manage_order, manage_return],
    system_prompt=SUPERVISOR_PROMPT
)

## Test 1 — Single-Domain Request

This test checks whether the supervisor can handle an order-tracking request.

In [19]:
response1 = supervisor_agent.invoke(
    {"messages": [HumanMessage(content="Track my order 1001")]}
)

for message in response1.get("messages", []):
    try:
        message.pretty_print()
    except:
        print(message)

================================ Human Message =================================

Track my order 1001
================================== Ai Message ==================================
Tool Calls:
  manage_order (call_69a06c4f6a624a80b5d0d28b)
 Call ID: call_69a06c4f6a624a80b5d0d28b
  Args:
    order_id: 1001
================================= Tool Message =================================
Name: manage_order

The order with ID 1001 has been shipped and is expected to arrive tomorrow. Let me know if you need further assistance!
================================== Ai Message ==================================

Your order 1001 has been shipped and is expected to arrive tomorrow. Let me know if you need further assistance!


## Test 2 — Multi-Domain Request

This test checks whether the supervisor can delegate different parts of the request to different sub-agents.

In [20]:
response2 = supervisor_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Track order 1001 and check if I can return headphones after 7 days."
            )
        ]
    }
)

for message in response2.get("messages", []):
    try:
        message.pretty_print()
    except:
        print(message)

================================ Human Message =================================

Track order 1001 and check if I can return headphones after 7 days.
================================== Ai Message ==================================
Tool Calls:
  manage_order (call_b2cfe7030aea41a18fcc87bd)
 Call ID: call_b2cfe7030aea41a18fcc87bd
  Args:
    order_id: 1001
================================= Tool Message =================================
Name: manage_order

The order with ID 1001 has been shipped and is expected to arrive tomorrow. Let me know if you need further assistance!
================================== Ai Message ==================================
Tool Calls:
  manage_return (call_ea46c62ec4834fa3b628cdc4)
 Call ID: call_ea46c62ec4834fa3b628cdc4
  Args:
    days_since_delivery: 7
    product_name: headphones
================================= Tool Message =================================
Name: manage_return

Yes—you can return the headphones because you’re still within the 14‑day re

## Why One Agent Is Not Enough

One agent is not enough for this task because online shopping support includes multiple domains.

- Order tracking requires shipment lookup.
- Returns require return policy checks.
- Return requests require a different action.

Each task uses different tools and different instructions.
Using the supervisor pattern is better because the supervisor can  assign each subtask to the appropriate specialized sub-agent, then combine the outputs into one final response.